# ER diagram → database schema
Draw your entities and relationships as text (Mermaid `erDiagram`); `erplay` generates the `CREATE TABLE` statements with primary and foreign keys and runs them.

This notebook uses its own database file (`work/er-to-schema.duckdb`), so it can be open next to `playground.ipynb`.

In [ ]:
import erplay
con = erplay.connect("er-to-schema")
erplay.reset(con)   # start clean

## The rules

| Mermaid | Meaning |
|---|---|
| `A ||--o{ B` | one A has zero or many B → table B gets a foreign key to A |
| `A |o--o{ B` | same, but the A is optional → the foreign key may be NULL |
| `A }o--o{ B` | many-to-many → a junction table `A_B` is created |
| `int id PK` | primary key (several `PK` columns = composite key) |
| `string email UK` | unique |
| `int owner_id FK` | name the foreign key column yourself (otherwise `<parent>_<pk>` is added) |

Types: `string`, `int`, `float`, `bool`, `date`, `datetime`, or any DuckDB type (`DECIMAL(10,2)`).

## Your diagram
Edit the text, run the cell, look at the generated SQL and the result.

In [ ]:
diagram = """
erDiagram
    AUTHOR ||--o{ BOOK : writes
    MEMBER ||--o{ LOAN : makes
    BOOK   ||--o{ LOAN : "is lent in"
    AUTHOR {
        int id PK
        string name
    }
    BOOK {
        int id PK
        string title
        int year
    }
    MEMBER {
        int id PK
        string name
        string email UK
    }
    LOAN {
        int id PK
        date loaned_on
    }
"""
erplay.reset(con)
erplay.apply_mermaid(con, diagram)

## Try it
The keys are enforced, e.g. a book needs an existing author:

In [ ]:
import duckdb
con.execute("INSERT INTO AUTHOR VALUES (1, 'Terry Pratchett')")
con.execute("INSERT INTO BOOK VALUES (1, 'Small Gods', 1992, 1)")          # ok
try:
    con.execute("INSERT INTO BOOK VALUES (2, 'Ghost Book', 2000, 99)")     # author 99 does not exist
except duckdb.ConstraintException as e:
    print("Rejected:", e)
con.sql("SELECT b.title, a.name FROM BOOK b JOIN AUTHOR a ON a.id = b.author_id").df()

## Many-to-many
`}o--o{` creates the junction table for you.

In [ ]:
erplay.reset(con)
erplay.apply_mermaid(con, """
erDiagram
    STUDENT }o--o{ COURSE : attends
    STUDENT {
        int id PK
        string name
    }
    COURSE {
        int id PK
        string title
    }
""")